# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Printing high-level metadata
print(f"Dataset Title: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}")
print(f"ID: {dataset.metadata.id}")
print(f"Version: {dataset.metadata.version}")


## 2. Data Overview
Review available record sets, fields, and their `@id`s.

We explore all available record sets, fields, and columns via their `@id` using the `mlcroissant` API.

In [ ]:
# List all available record sets and their fields by their '@id'.

record_sets = list(dataset.record_sets)
if not record_sets:
    print("No RecordSets declared in the top-level metadata; trying automatic inference...")
    # mlcroissant will still provide record_sets dynamically if files are present
    record_sets = [rs for rs in dataset._record_sets]
else:
    record_sets = [rs.id for rs in dataset.record_sets]

print("Available Record Sets:")
for rs in dataset.record_sets:
    print(f"  - @id: {rs.id}, name: {rs.name}")

print("\nPreviewing fields for each record set:")
for rs in dataset.record_sets:
    print(f"\nRecord Set: {rs.id}")
    for field in rs.fields:
        print(f"  Field @id: {field.id}, name: {field.name}, dataType: {field.data_type}")


## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview section.

In [ ]:
# Extract data from each record set by @id and preview column @ids
dataframes = {}

for rs in dataset.record_sets:
    rs_id = rs.id
    try:
        records = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Loaded RecordSet: {rs_id} with {len(df)} records.")
        print(f"Columns (@id): {list(df.columns)}\n")
    except Exception as e:
        print(f"Skipping RecordSet {rs_id}: {e}")

# For demonstration, select the first non-empty record set
main_rs_id = None
for rs_id, df in dataframes.items():
    if not df.empty:
        main_rs_id = rs_id
        break
if main_rs_id:
    print(f"First non-empty RecordSet for analysis: {main_rs_id}")
    display(dataframes[main_rs_id].head())
else:
    print("No non-empty RecordSets available.")


## 4. Exploratory Data Analysis (EDA)

Apply data processing steps such as filtering, normalizing, and grouping using `@id` of fields. We'll select a numeric field, filter for some threshold, normalize it, and group by a categorical field if available.

In [ ]:
# Select a numeric field and a grouping field by their @id

import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Identify potential numeric and group fields
df = dataframes[main_rs_id]

# For demonstration, let's attempt to detect some likely columns based on typical medical variables
possible_numeric_fields = [col for col in df.columns if any(s in col.lower() for s in ['age', 'interval', 'count', 'number']) or df[col].dtype in [np.int64, np.float64]]
print(f"Numeric fields detected: {possible_numeric_fields}")
if possible_numeric_fields:
    numeric_field = possible_numeric_fields[0]
else:
    numeric_field = df.select_dtypes(include=[np.number]).columns[0] if len(df.select_dtypes(include=[np.number]).columns) else df.columns[0]

# Grouping field: look for typical categorical variables (like sex or anatomical location)
possible_group_fields = [col for col in df.columns if any(s in col.lower() for s in ['sex', 'gender', 'location', 'msi', 'anatomical', 'status'])]
group_field = possible_group_fields[0] if possible_group_fields else df.columns[1]

print(f"Using numeric field '@id': {numeric_field}")
print(f"Using grouping field '@id': {group_field}")

# Drop missing values for the numeric field
filtered_df = df.dropna(subset=[numeric_field]).copy()

# Try filtering for records with value > threshold (choose a representative threshold or 0 if not appropriate)
threshold = filtered_df[numeric_field].median()
filtered_df = filtered_df[filtered_df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold}:")
display(filtered_df.head())

# Normalize the numeric field (z-score)
norm_col = f"{numeric_field}_normalized"
filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()

print(f"Normalized {numeric_field} for filtered records:")
display(filtered_df[[numeric_field, norm_col]].head())

if group_field in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
    print(f"Grouped data by {group_field} (mean of {numeric_field}):")
    display(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

We'll plot the distribution of the selected numeric field and show mean values by group.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field (normalized)
plt.figure(figsize=(7,4))
sns.histplot(filtered_df[norm_col], kde=True, bins=15, color='skyblue')
plt.title(f"Distribution of Normalized {numeric_field} (@id: {numeric_field})")
plt.xlabel(f"{numeric_field} (normalized)")
plt.ylabel('Count')
plt.show()

# If grouping field available, plot grouped means
if group_field in filtered_df.columns:
    plt.figure(figsize=(9,4))
    sns.barplot(data=grouped_df, x=group_field, y=numeric_field, palette='viridis')
    plt.title(f"Mean of {numeric_field} by {group_field}")
    plt.xlabel(group_field)
    plt.ylabel(f"Mean {numeric_field}")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

## 6. Conclusion

- This notebook demonstrated how to load, explore, and analyze the FAIR² clinical dataset using the Croissant schema and the `mlcroissant` library.
- Record sets, fields, and columns were referenced by their `@id`, ensuring reproducibility and schema traceability.
- We showed basic EDA, including filtering on a numeric variable, normalization, grouping, and data visualization.
- You may further extend this notebook for more domain-specific analyses (e.g., statistical testing, machine learning, immuno-oncology subgroup exploration, etc.).
